In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import warnings
import sys
import seaborn as sns
import matplotlib.pyplot as plt

sys.path.append(str(Path("../src").resolve()))

from config import Config
from dataloader import load_data_csv
from preprocessor import Preprocessor

warnings.simplefilter(action="ignore", category=FutureWarning)

config_path = Path("../config.toml").resolve()
config = Config.load(config_path)
df = load_data_csv(config)

preprocessor = Preprocessor(config)
df = preprocessor.run()
df

In [ ]:
# message length
import emoji


df['message_length'] = df['message'].astype(str).apply(len)

# has emoji
def count_emojis(text):
    return len([char for char in str(text) if char in emoji.EMOJI_DATA])

df['emoji_count'] = df['message'].apply(count_emojis)

# hour
df['hour'] = df['timestamp'].dt.hour

df


In [ ]:
# find correlaties
corr = df[['age', 'emoji_count', 'message_length', 'hour']].corr()

plt.figure(figsize=(5, 4))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", square=True)
plt.title('Correlatiematrix')
plt.tight_layout()
plt.show()


In [ ]:
# Bereken aantal berichten per persoon tijdens carnaval en erbuiten
carnaval_counts = df.groupby(['author', 'is_carnaval']).size().unstack(fill_value=0)
carnaval_counts['age'] = df.groupby('author')['age'].first()
carnaval_counts['carnaval_ratio'] = carnaval_counts[True] / (carnaval_counts[True] + carnaval_counts[False])

plt.figure(figsize=(8, 6))
sns.scatterplot(data=carnaval_counts, x='age', y='carnaval_ratio')
sns.regplot(data=carnaval_counts, x='age', y='carnaval_ratio')
plt.xlabel('Leeftijd')
plt.ylabel('Proportie berichten tijdens carnaval')
plt.title('Zijn jongere familieleden actiever tijdens carnaval?')
plt.tight_layout()
plt.show()


De scatterplot laat zien dat familieleden van rond de 35 jaar relatief het meest actief zijn tijdens carnaval: bij hen is ~6% van hun berichten in de carnavalsweek verstuurd. Zowel jongere als oudere familieleden lijken iets minder ‘aan’ te gaan tijdens carnaval. Dit kan duiden op een centrale rol van de 30–40-jarige generatie in het organiseren of beleven van het feest.

In [ ]:
# Leeftijd per persoon
ages = df.groupby('author')['age'].first()

# Groepeer leeftijden in bins
bins = [0, 30, 40, 50, 100]
labels = ['<30', '30–39', '40–49', '50+']
age_groups = pd.cut(ages, bins=bins, labels=labels)

# Tel hoeveel personen in elke leeftijdsgroep zitten
age_group_counts = age_groups.value_counts().sort_index()


plt.figure(figsize=(6, 5))
age_group_counts.plot(kind='bar')
plt.xlabel('Leeftijdsgroep')
plt.ylabel('Aantal personen')
plt.title('Verdeling van leeftijden in leeftijdsgroepen')
plt.tight_layout()
plt.show()


De groep tussen 30 en 40 jaar is oververtegenwoordigd in de chat, waardoor het logisch is dat zij ook het meest actief lijken tijdens carnaval. De verhouding carnaval / totaal is daarom niet volledig onafhankelijk van groepssamenstelling.

In [ ]:
# nieuw DataFrame op persoonsniveau
person_df = df.groupby('author').agg({
    'emoji_count': 'mean',
    'message_length': 'mean',
    'age': 'first',
    'gender': 'first',
    'is_inlaw': 'first',
    'is_carnaval': 'mean',  # Hoeveel % van hun berichten zijn tijdens carnaval
    'timestamp': 'count'    # Aantal berichten 
}).rename(columns={'timestamp': 'message_count'}).reset_index()

# Haal NAN weg
person_df = person_df[person_df['gender'].notna()].copy()

# Zet gender om naar numeriek
person_df['gender_num'] = person_df['gender'].str.lower().map({'m': 0, 'f': 1})

# Alleen numerieke kolommen gebruiken
corr = person_df[['age', 'emoji_count', 'message_length', 'message_count', 'is_carnaval','gender_num','is_inlaw']].corr()

plt.figure(figsize=(6,5))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlaties tussen persoonskenmerken en gedrag")
plt.tight_layout()
plt.show()

## 🎭 Carnaval en gedrag

**Sterke negatieve correlatie tussen `is_inlaw` en `is_carnaval` (-0.64)**  
→ Aangetrouwde familieleden zijn veel minder actief tijdens carnaval.  
*Logisch als het een ‘carnavalsfamilie’ is en de ‘in-laws’ niet uit de regio komen of minder betrokken zijn.*

**Sterke negatieve correlatie tussen `gender_num` (vrouw = 1) en `is_carnaval` (-0.63)**  
→ Vrouwen zijn minder actief tijdens carnaval dan mannen.  
*Zou te maken kunnen hebben met dezelfde reden als hierboven: veel vrouwen zijn aangetrouwd.*

**Sterke positieve correlatie tussen `message_count` en `is_carnaval` (0.60)**  
→ Hoe meer iemand over het algemeen stuurt, hoe groter het aandeel tijdens carnaval.  
*Dus de meest actieve mensen zijn ook tijdens carnaval het meest aanwezig.*

---

## 🧑‍🤝‍🧑 Aangetrouwde familieleden (`is_inlaw`)

**Sterke negatieve correlatie met `message_count` (-0.81)**  
→ Aangetrouwden sturen over het algemeen véél minder berichten.  
*Ze zijn er wel bij, maar minder actief.*

**Negatieve correlatie met `message_length` (-0.22)**  
→ Ze sturen gemiddeld ook iets kortere berichten.

**Positieve correlatie met `gender_num` (0.33)**  
→ Aangetrouwden zijn vaker vrouw.


In [ ]:
# Zijn niet aangetrouwde leden actiever tijdens carnaval?

# Zet 0/1 om naar 'Nee'/'Ja'
person_df['is_inlaw_label'] = person_df['is_inlaw'].map({0: 'Nee', 1: 'Ja'})

plt.figure(figsize=(6, 5))
sns.boxplot(
    data=person_df,
    x='is_inlaw_label',
    y='is_carnaval',
    palette={"Nee": "skyblue", "Ja": "lightcoral"}
)
plt.xlabel('Is aangetrouwd?')
plt.ylabel('Aandeel berichten tijdens carnaval')
plt.title('Carnavalactiviteit aangetrouwd of niet')
plt.tight_layout()
plt.show()

## 🎭 Carnavalactiviteit: aangetrouwd of niet?

Uit de boxplot blijkt duidelijk dat **niet-aangetrouwde familieleden** een groter aandeel van hun berichten tijdens carnaval versturen dan aangetrouwde familieleden:

- Het **mediaan aandeel** berichten tijdens carnaval ligt bij niet-aangetrouwden rond de **0.045**, tegenover ongeveer **0.02–0.025** bij aangetrouwden.
- Niet-aangetrouwden tonen een grotere **spreiding**, wat erop wijst dat sommigen extreem actief zijn in deze periode.
- Aangetrouwden zijn gemiddeld duidelijk **minder betrokken** in de carnaval-chatactiviteit.

**Conclusie**: In deze typische carnavalsfamilie zijn het vooral de **kernleden (niet-aangetrouwden)** die de meeste activiteit vertonen tijdens carnaval. Aangetrouwden doen wel mee, maar lijken iets meer op de achtergrond te blijven.

In [ ]:
# Zijn niet aangetrouwde leden uberhaubt actiever?

# Zet 0/1 om naar 'Nee'/'Ja'
person_df['is_inlaw_label'] = person_df['is_inlaw'].map({0: 'Nee', 1: 'Ja'})

plt.figure(figsize=(6, 5))
sns.boxplot(
    data=person_df,
    x='is_inlaw_label',
    y='message_count',
    palette={"Nee": "skyblue", "Ja": "lightcoral"}
)
plt.xlabel('Is aangetrouwd?')
plt.ylabel('Aandeel berichten in groepsapp')
plt.title('Chat activiteit aangetrouwd of niet')
plt.tight_layout()
plt.show()